In [1]:
import cv2
import numpy as np
import os
import sys
import shutil
from PIL import Image
from paddle.vision.transforms import functional as F
import re
from ast import literal_eval
from PIL import ImageDraw, ImageFont
import matplotlib.pyplot as plt

In [2]:
source_imgs_dir = 'C:\\Users\\hp\\Desktop\\imgs1_b1\\'
_source_name_lists = os.listdir(source_imgs_dir)

d:\Anaconda\envs\paddle_env\lib\site-packages\ipykernel\ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [3]:
imgs_name = []
for _item in _source_name_lists:
    if _item.split('.')[-1] in ['jpg', 'png', 'bmp', 'jpeg']:
        imgs_name.append(_item)

In [4]:
# 读取文本文件
_path = os.path.join(source_imgs_dir, 'Label.txt')
with open(_path, 'r', encoding='utf-8') as f:
    contents_label = f.readlines()
_path = os.path.join(source_imgs_dir, 'fileState.txt')
with open(_path, 'r', encoding='utf-8') as f:
    contents_state = f.readlines()


In [5]:
print((contents_label[0].split('\t')[0]).split('/')[-1], (contents_label[0].split('\t')[-1]))

1.jpg [{"transcription": "038: 01/0", "points": [[656, 291], [941, 291], [941, 350], [656, 350]], "difficult": false}, {"transcription": "x", "points": [[604, 348], [643, 348], [643, 387], [604, 387]], "difficult": false}, {"transcription": "x", "points": [[841, 348], [879, 348], [879, 381], [841, 381]], "difficult": false}, {"transcription": "20° 96.4 96.4", "points": [[333, 392], [913, 385], [914, 462], [330, 475]], "difficult": false}, {"transcription": "60° 97.6 97.6", "points": [[333, 562], [921, 550], [913, 472], [331, 480]], "difficult": false}, {"transcription": "85° 99.7 99.7", "points": [[327, 572], [922, 561], [924, 644], [328, 656]], "difficult": false}]



In [6]:
print(contents_state[0])

C:\Users\hp\Desktop\imgs_line\1.jpg	1



In [5]:
# 把标签中的图片和其索引对应起来，方便后面使用
img_to_label_key = {}
for _i in range(0, len(contents_label)):
    img_to_label_key[(contents_label[_i].split('\t')[0]).split('/')[-1]] = _i

In [6]:
# 图像增强
_sub_dir = 'imgs1_b1/'
_label_file_name = "Label_new.txt"
_state_file_name = "fileState_new.txt"
with open(os.path.join(source_imgs_dir,_label_file_name),'wb') as f1, open(os.path.join(source_imgs_dir,_state_file_name),'wb') as f2:
    # 先把已有的标签写进去
    for item in contents_label:
        f1.write(item.encode())
    for item in contents_state:
        f2.write(item.encode())
    #示例性地写一个文件
    # f1.write(('imgs1_b1/0_0.jpg'+'\t'+contents_label[0].split('\t')[-1]).encode())
    # f2.write((os.path.join(source_imgs_dir,'0_0.jpg')+'\t'+'1'+'\n').encode())
    
    #数据增强
    per_num = 100
    dist_arr = np.linspace(1, 100, num=100) #随机选择的数组
    bright_arr = np.hstack((np.linspace(0.4,0.99, 100), np.linspace(1.1,1.5,30))) # 变暗的多，变亮的少
    for _imgName in imgs_name:
        for i in range(0,per_num):
            img = Image.open(os.path.join(source_imgs_dir, _imgName))#加载图片
            # 亮度变换最主要
            _bright_factor = np.random.choice(bright_arr)
            converted_img = F.adjust_brightness(img, _bright_factor)
            # 随机选择一个数决定是否使用下面的变换
            _flag = np.random.choice(dist_arr) 
            if _flag>=70.:
                #对比度
                _contrast_factor = np.random.choice([0.4,0.45,0.5,0.55,0.6,0.65,0.7,0.75,0.8,0.9,1.2,1.3,1.4,1.5,1.6])
                converted_img = F.adjust_contrast(converted_img,_contrast_factor)

                # 饱和度
                _saturation_factor = np.random.choice([0.4,0.45,0.5,0.55,0.6,0.65,0.7,0.75,0.8,0.9,0.91,0.92,0.93,0.94,0.95,0.96])
                converted_img = F.adjust_saturation(converted_img, _saturation_factor)
            if _flag > 90:
                # 色调
                _hue_factor = np.random.choice([-0.3,-0.2,-0.15,-0.1,-0.05,-0.02,0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.08
                                                ,0.09,0.1,0.11,0.12,0.13,0.14,0.15,0.2,0.3])
                converted_img = F.adjust_hue(converted_img,_hue_factor) #[-0.5,0.5]
            # 保存新图片和标签
            _new_img_name = _imgName.split('.')[0]+'_'+str(i)+'.'+_imgName.split('.')[-1]
            converted_img.save(os.path.join(source_imgs_dir, _new_img_name))
            
            _index = img_to_label_key[_imgName]
            f1.write((_sub_dir+_new_img_name+'\t'+contents_label[_index].split('\t')[-1]).encode())
            f2.write((os.path.join(source_imgs_dir,_new_img_name)+'\t'+'1'+'\n').encode())





d:\Anaconda\envs\paddle_env\lib\site-packages\ipykernel\ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
import random
fake_img = Image.open(os.path.join(source_imgs_dir, imgs_name[0]))

converted_img = F.adjust_brightness(fake_img, 1.)
converted_img = F.adjust_contrast(converted_img,2)
converted_img = F.adjust_hue(converted_img,0.) #[-0.5,0.5]
_factor = np.random.ranf()
_saturation_factor = np.random.choice([0.4,0.45,0.5,0.55,0.6,0.65,0.7,0.75,0.8,0.9,0.91,0.92,0.93,0.94,0.95,0.96])
print(_factor, _saturation_factor)
converted_img = F.adjust_saturation(converted_img, _factor)

converted_img.save(os.path.join(source_imgs_dir, '0_0.jpg'))
plt.imshow(converted_img)
print(converted_img.size)


In [ ]:
# print(np.random.randint(low=1, high=50, size=100))
# print(np.random.choice(np.linspace(1, 100, num=100)))

# print(np.linspace(0.4,0.99, 100))
np.hstack((np.linspace(0.4,0.99, 100), np.linspace(1.1,1.5,30)))

# np.hstack((np.array([0,1]), np.array([2,3])))